In [3]:
import pandas as pd

# Lê apenas as colunas necessárias
ratings = pd.read_csv("ml-latest-small/ratings.csv", usecols=["movieId"])
tags = pd.read_csv("ml-latest-small/tags.csv", usecols=["movieId", "tag"])

# Filtra as tags apenas para os filmes que estão em ratings
tags_filtradas = tags[tags["movieId"].isin(ratings["movieId"].unique())]

# Limita a no máximo 5 tags por filme
tags_limitadas = (
    tags_filtradas
    .groupby("movieId")
    .head(5)  # pega as 5 primeiras tags de cada filme
    .reset_index(drop=True)
)

# (Opcional) salva o resultado
tags_limitadas.to_csv("tags_filmes_filtradas.csv", index=False)

print(tags_limitadas.head(15))


    movieId                tag
0     60756              funny
1     60756    Highly quotable
2     60756       will ferrell
3     89774       Boxing story
4     89774                MMA
5     89774          Tom Hardy
6    106782              drugs
7    106782  Leonardo DiCaprio
8    106782    Martin Scorsese
9     48516       way too long
10      431          Al Pacino
11      431           gangster
12      431              mafia
13     1221          Al Pacino
14     1221              Mafia


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sentence_transformers import SentenceTransformer
import pickle

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ===========================
# 1. Lê CSVs
# ===========================
ratings = pd.read_csv("ratings_com_tags.csv")
movies = pd.read_csv("movies.csv")
tags = pd.read_csv("tags_filmes_filtradas.csv").dropna(subset=['tag'])

# Filtra apenas filmes presentes em ratings
ratings = ratings[ratings['movieId'].isin(movies['movieId'])].copy()
movies_in_ratings = movies[movies['movieId'].isin(ratings['movieId'].unique())].copy()
tags = tags[tags['movieId'].isin(ratings['movieId'].unique())]

print(f"Número de avaliações: {len(ratings)}")
print(f"Número de filmes: {len(movies_in_ratings)}")
print(f"Número de tags: {len(tags)}")
print(f"Número de usuários: {ratings['userId'].nunique()}")


c:\Users\Leoso\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.13.0+cpu
Device: cpu
Número de avaliações: 100122
Número de filmes: 9314
Número de tags: 2808
Número de usuários: 610


In [5]:
# ===========================
# 2. Processa gêneros
# ===========================
all_genres = sorted(set(g for gs in movies_in_ratings['genres'].dropna() for g in gs.split('|')))
genre2idx = {g: i for i, g in enumerate(all_genres)}

def get_genre_vector(genres_str):
    vec = np.zeros(len(all_genres), dtype=np.float32)
    if pd.notna(genres_str):
        for g in genres_str.split('|'):
            if g in genre2idx:
                vec[genre2idx[g]] = 1.0
    return vec

movies_in_ratings['genre_vec'] = movies_in_ratings['genres'].apply(get_genre_vector)

# ===========================
# 3. Mapeia IDs para índices contínuos
# ===========================
user2idx = {u: i for i, u in enumerate(sorted(ratings['userId'].unique()))}
movie2idx = {m: i for i, m in enumerate(sorted(ratings['movieId'].unique()))}

ratings['user_idx'] = ratings['userId'].map(user2idx)
ratings['movie_idx'] = ratings['movieId'].map(movie2idx)

# Dicionário de gêneros por movie_idx
movie_idx_to_genres = {movie2idx[row['movieId']]: row['genre_vec']
                       for _, row in movies_in_ratings.iterrows()
                       if row['movieId'] in movie2idx}

print(f"Número de gêneros: {len(all_genres)}")
print(f"Exemplo de gêneros: {all_genres[:10]}")
print(f"Número de usuários mapeados: {len(user2idx)}")
print(f"Número de filmes mapeados: {len(movie2idx)}")


Número de gêneros: 20
Exemplo de gêneros: ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy']
Número de usuários mapeados: 610
Número de filmes mapeados: 9314


In [6]:
# ===========================
# 4. Processa tags com embeddings pré-calculados
# ===========================
print("Carregando modelo de embeddings de tags...")
model_st = SentenceTransformer("all-MiniLM-L6-v2")

# Calcula embeddings para cada tag única (cache para evitar duplicatas)
unique_tags = tags['tag'].unique()
tag_embedding_cache = {}
for tag in unique_tags:
    tag_embedding_cache[tag] = model_st.encode(tag, normalize_embeddings=True)

# Mapeia embeddings para cada filme
movie_tag_embs = {}
for movie_id in tags['movieId'].unique():
    movie_tags = tags[tags['movieId'] == movie_id]['tag'].values
    embs = np.array([tag_embedding_cache[t] for t in movie_tags])
    movie_tag_embs[movie_id] = np.mean(embs, axis=0)  # média dos embeddings

print(f"Embeddings de tags calculados para {len(movie_tag_embs)} filmes")

# ===========================
# 5. Combina gênero + tag com pré-convertidos em tensores
# ===========================
TAG_DIM = 384  # Dimensão do MiniLM
movie_idx_to_features = {}

for _, row in movies_in_ratings.iterrows():
    mid = row["movieId"]
    if mid in movie2idx:
        genre_vec = movie_idx_to_genres.get(movie2idx[mid], np.zeros(len(all_genres), dtype=np.float32))
        tag_vec = movie_tag_embs.get(mid, np.zeros(TAG_DIM, dtype=np.float32))
        
        # Pré-converte para tensor (OTIMIZAÇÃO)
        movie_idx_to_features[movie2idx[mid]] = {
            "genre_tensor": torch.tensor(genre_vec, dtype=torch.float32),
            "tag_tensor": torch.tensor(tag_vec, dtype=torch.float32)
        }

print(f"Número de filmes com features pré-vetorizadas: {len(movie_idx_to_features)}")
print(f"TAG_DIM: {TAG_DIM}")


Carregando modelo de embeddings de tags...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2621.30it/s]


Embeddings de tags calculados para 1546 filmes
Número de filmes com features pré-vetorizadas: 9314
TAG_DIM: 384


In [7]:
# ===========================
# 6. Dataset Otimizado com Tensores Pré-vetorizados
# ===========================
class RatingDataset(Dataset):
    """Dataset otimizado com features pré-convertidas em tensores."""
    def __init__(self, df, movie_features, n_genres, tag_dim, device='cpu'):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.movies = torch.tensor(df['movie_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
        self.n_genres = n_genres
        self.tag_dim = tag_dim
        self.movie_features = movie_features
        self.device = device

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        m = self.movies[idx].item()
        if m in self.movie_features:
            genre_tensor = self.movie_features[m]["genre_tensor"]
            tag_tensor = self.movie_features[m]["tag_tensor"]
        else:
            genre_tensor = torch.zeros(self.n_genres, dtype=torch.float32)
            tag_tensor = torch.zeros(self.tag_dim, dtype=torch.float32)
        
        return (
            self.users[idx],
            self.movies[idx],
            genre_tensor,
            tag_tensor,
            self.ratings[idx]
        )

# Split treino/validação/teste (80% treino, 10% validação, 10% teste)
train_df, temp_df = train_test_split(ratings, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_ds = RatingDataset(train_df, movie_idx_to_features, len(all_genres), TAG_DIM, device=device)
val_ds = RatingDataset(val_df, movie_idx_to_features, len(all_genres), TAG_DIM, device=device)
test_ds = RatingDataset(test_df, movie_idx_to_features, len(all_genres), TAG_DIM, device=device)

BATCH_SIZE = 512
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Dataset splits:")
print(f"  Treino: {len(train_df)} ({len(train_df)/len(ratings)*100:.1f}%)")
print(f"  Validação: {len(val_df)} ({len(val_df)/len(ratings)*100:.1f}%)")
print(f"  Teste: {len(test_df)} ({len(test_df)/len(ratings)*100:.1f}%)")


Dataset splits:
  Treino: 80097 (80.0%)
  Validação: 10012 (10.0%)
  Teste: 10013 (10.0%)


In [9]:
# ===========================
# 7. Modelo com Melhorias Arquiteturais
# ===========================

class MFModel(nn.Module):
    """Modelo com:
    - Bias global para centralizar a predição
    - Normalização simétrica de usuário e filme
    - MLP para fusão de features em vez de linear simples
    """
    def __init__(self, n_users, n_movies, n_genres, tag_dim,
                 emb_dim=64, genre_emb_dim=16, tag_emb_dim=32, dropout=0.1):
        super().__init__()
        
        # Embeddings
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        
        # Biases
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        
        # NOVO: Bias global para centralizar a predição
        self.global_bias = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Processamento de gêneros e tags
        self.genre_fc = nn.Sequential(
            nn.Linear(n_genres, genre_emb_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.tag_fc = nn.Sequential(
            nn.Linear(tag_dim, tag_emb_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # NOVO: MLP para fusão em vez de linear simples
        combined_dim = emb_dim + genre_emb_dim + tag_emb_dim
        self.movie_combine_mlp = nn.Sequential(
            nn.Linear(combined_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, emb_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, users, movies, genres, tags):
        # Embeddings
        u = self.user_emb(users)  # (batch, emb_dim)
        v = self.movie_emb(movies)  # (batch, emb_dim)
        g = self.genre_fc(genres)  # (batch, genre_emb_dim)
        t = self.tag_fc(tags)  # (batch, tag_emb_dim)
        
        # Dropout
        u = self.dropout(u)
        v = self.dropout(v)
        g = self.dropout(g)
        t = self.dropout(t)
        
        # Combina features do filme com MLP
        v_combined = self.movie_combine_mlp(torch.cat([v, g, t], dim=1))  # (batch, emb_dim)
        
        # NOVO: Normaliza AMBOS usuário e filme (simetria)
        u_norm = F.normalize(u, p=2, dim=1)
        v_norm = F.normalize(v_combined, p=2, dim=1)
        
        # Produto escalar = cosseno quando normalizados
        dot_product = (u_norm * v_norm).sum(dim=1)  # (batch,)
        
        # Predição: global_bias + user_bias + movie_bias + dot_product
        pred = (
            self.global_bias +
            self.user_bias(users).squeeze(-1) +
            self.movie_bias(movies).squeeze(-1) +
            dot_product
        )
        return pred
    
    def get_movie_embedding(self, movies, genres, tags):
        """Retorna embedding normalizado do filme para exportação."""
        v = self.movie_emb(movies)
        g = self.genre_fc(genres)
        t = self.tag_fc(tags)
        v_combined = self.movie_combine_mlp(torch.cat([v, g, t], dim=1))
        v_norm = F.normalize(v_combined, p=2, dim=1)
        return v_norm
    
    def get_user_embedding(self, users):
        """Retorna embedding normalizado do usuário para exportação."""
        u = self.user_emb(users)
        u_norm = F.normalize(u, p=2, dim=1)
        return u_norm

# ===========================
# 8. Inicializa modelo e otimizador
# ===========================
n_users = len(user2idx)
n_movies = len(movie2idx)
n_genres = len(all_genres)

model = MFModel(n_users, n_movies, n_genres, TAG_DIM,
                emb_dim=64, genre_emb_dim=16, tag_emb_dim=32, dropout=0.1)
model = model.to(device)

# NOVO: weight_decay para regularização L2
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

# NOVO: Scheduler para reduzir LR quando loss não melhora
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

criterion = nn.MSELoss()

print(f"Modelo criado com {sum(p.numel() for p in model.parameters())} parâmetros")
print(f"Device: {device}")


Modelo criado com 680437 parâmetros
Device: cpu


In [10]:
# ===========================
# 9. Treino com Early Stopping e Validação por Época
# ===========================

MAX_EPOCHS = 50
PATIENCE = 5  # Para early stopping
best_val_loss = float('inf')
patience_counter = 0
train_losses = []
val_losses = []

for epoch in range(MAX_EPOCHS):
    # ===== TREINO =====
    model.train()
    train_loss = 0.0
    
    for users, movies, genres, tags_batch, ratings_batch in train_dl:
        users = users.to(device)
        movies = movies.to(device)
        genres = genres.to(device)
        tags_batch = tags_batch.to(device)
        ratings_batch = ratings_batch.to(device)
        
        preds = model(users, movies, genres, tags_batch)
        loss = criterion(preds, ratings_batch)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_dl)
    train_losses.append(avg_train_loss)
    
    # ===== VALIDAÇÃO =====
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for users, movies, genres, tags_batch, ratings_batch in val_dl:
            users = users.to(device)
            movies = movies.to(device)
            genres = genres.to(device)
            tags_batch = tags_batch.to(device)
            ratings_batch = ratings_batch.to(device)
            
            preds = model(users, movies, genres, tags_batch)
            loss = criterion(preds, ratings_batch)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_dl)
    val_losses.append(avg_val_loss)
    
    # ===== EARLY STOPPING E SCHEDULER =====
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Salva melhor modelo
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
    
    # Reduz LR se loss não melhora
    scheduler.step(avg_val_loss)
    
    # Print a cada epoch
    print(f"Epoch {epoch+1:2d}/{MAX_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Patience: {patience_counter}/{PATIENCE}")
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n✅ Early stopping em epoch {epoch+1}")
        break

# Carrega o melhor modelo
model.load_state_dict(torch.load('best_model.pth'))
print(f"\n✅ Melhor modelo restaurado (val_loss: {best_val_loss:.4f})")


Epoch  1/50 | Train Loss: 9.9994 | Val Loss: 5.3263 | Patience: 0/5
Epoch  2/50 | Train Loss: 3.0490 | Val Loss: 1.5967 | Patience: 0/5
Epoch  3/50 | Train Loss: 1.2311 | Val Loss: 1.0093 | Patience: 0/5
Epoch  4/50 | Train Loss: 0.8797 | Val Loss: 0.8509 | Patience: 0/5
Epoch  5/50 | Train Loss: 0.7681 | Val Loss: 0.7887 | Patience: 0/5
Epoch  6/50 | Train Loss: 0.7115 | Val Loss: 0.7642 | Patience: 0/5
Epoch  7/50 | Train Loss: 0.6731 | Val Loss: 0.7501 | Patience: 0/5
Epoch  8/50 | Train Loss: 0.6402 | Val Loss: 0.7417 | Patience: 0/5
Epoch  9/50 | Train Loss: 0.6133 | Val Loss: 0.7445 | Patience: 1/5
Epoch 10/50 | Train Loss: 0.5916 | Val Loss: 0.7414 | Patience: 0/5
Epoch 11/50 | Train Loss: 0.5752 | Val Loss: 0.7438 | Patience: 1/5
Epoch 12/50 | Train Loss: 0.5612 | Val Loss: 0.7371 | Patience: 0/5
Epoch 13/50 | Train Loss: 0.5490 | Val Loss: 0.7401 | Patience: 1/5
Epoch 14/50 | Train Loss: 0.5406 | Val Loss: 0.7352 | Patience: 0/5
Epoch 15/50 | Train Loss: 0.5314 | Val Loss: 0.7

In [11]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ===========================
# 10. Avaliação no Conjunto de Teste
# ===========================

model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for users, movies, genres, tags_batch, ratings_batch in test_dl:
        users = users.to(device)
        movies = movies.to(device)
        genres = genres.to(device)
        tags_batch = tags_batch.to(device)
        
        preds = model(users, movies, genres, tags_batch)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(ratings_batch.cpu().numpy())

# Converte para arrays numpy
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Calcula métricas
rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
mae = mean_absolute_error(all_targets, all_preds)

print("\n" + "="*50)
print("AVALIAÇÃO NO CONJUNTO DE TESTE")
print("="*50)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"Predições (min, max, mean): {all_preds.min():.2f}, {all_preds.max():.2f}, {all_preds.mean():.2f}")
print(f"Targets (min, max, mean): {all_targets.min():.2f}, {all_targets.max():.2f}, {all_targets.mean():.2f}")



AVALIAÇÃO NO CONJUNTO DE TESTE
RMSE: 0.8476
MAE:  0.6472
Predições (min, max, mean): 0.87, 5.33, 3.52
Targets (min, max, mean): 0.50, 5.00, 3.49


In [12]:
# ===========================
# 11. Exportação de Embeddings com Normalização Consistente
# ===========================

def salvar_embeddings_completos(model, movie_idx_to_features, n_users, n_movies,
                                n_genres, tag_dim, user2idx_map, movie2idx_map, device):
    """Salva embeddings normalizados de forma consistente.
    
    Garante que:
    - Usuário e filme são ambos normalizados (simetria)
    - Usam a mesma transformação (normalize com p=2, dim=1)
    """
    print("Exportando embeddings normalizados...")
    
    model.eval()
    
    # -------------------
    # 1. Usuários (normalizados)
    # -------------------
    user_ids_tensor = torch.arange(n_users, dtype=torch.long).to(device)
    with torch.no_grad():
        user_embs = model.get_user_embedding(user_ids_tensor)
    user_embs = user_embs.cpu().numpy()  # (n_users, emb_dim), normalized
    
    # -------------------
    # 2. Filmes (normalizados)
    # -------------------
    movie_embs_list = []
    for movie_idx in range(n_movies):
        if movie_idx in movie_idx_to_features:
            feats = movie_idx_to_features[movie_idx]
            genre_tensor = feats["genre_tensor"].unsqueeze(0).to(device)
            tag_tensor = feats["tag_tensor"].unsqueeze(0).to(device)
        else:
            genre_tensor = torch.zeros(1, n_genres, dtype=torch.float32).to(device)
            tag_tensor = torch.zeros(1, tag_dim, dtype=torch.float32).to(device)
        
        movie_tensor = torch.tensor([movie_idx], dtype=torch.long).to(device)
        
        with torch.no_grad():
            emb = model.get_movie_embedding(movie_tensor, genre_tensor, tag_tensor)
        movie_embs_list.append(emb.cpu().numpy())
    
    movie_embs = np.vstack(movie_embs_list)  # (n_movies, emb_dim), normalized
    
    # -------------------
    # 3. Salva arquivos
    # -------------------
    np.save("user_embeddings.npy", user_embs)
    np.save("movie_embeddings.npy", movie_embs)
    
    with open("user2idx.pkl", "wb") as f:
        pickle.dump(user2idx_map, f)
    
    with open("movie2idx.pkl", "wb") as f:
        pickle.dump(movie2idx_map, f)
    
    # Salva metadados
    metadata = {
        "n_users": n_users,
        "n_movies": n_movies,
        "emb_dim": user_embs.shape[1],
        "n_genres": n_genres,
        "tag_dim": tag_dim,
        "normalization": "L2 (cosine similarity)",
    }
    
    import json
    with open("embeddings_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print(f"\n✅ Embeddings salvos com sucesso!")
    print(f"   user_embeddings.npy: {user_embs.shape}")
    print(f"   movie_embeddings.npy: {movie_embs.shape}")
    print(f"   user2idx.pkl: {len(user2idx_map)} usuários")
    print(f"   movie2idx.pkl: {len(movie2idx_map)} filmes")
    print(f"   embeddings_metadata.json: metadados")
    
    return user_embs, movie_embs

# Executa exportação
user_embs, movie_embs = salvar_embeddings_completos(
    model,
    movie_idx_to_features,
    n_users, n_movies,
    n_genres, TAG_DIM,
    user2idx, movie2idx,
    device
)

# Verifica normalização
user_norms = np.linalg.norm(user_embs, axis=1)
movie_norms = np.linalg.norm(movie_embs, axis=1)
print(f"\n📊 Verificação de Normalização:")
print(f"   User embeddings normas (deve ser ~1.0): min={user_norms.min():.6f}, max={user_norms.max():.6f}, mean={user_norms.mean():.6f}")
print(f"   Movie embeddings normas (deve ser ~1.0): min={movie_norms.min():.6f}, max={movie_norms.max():.6f}, mean={movie_norms.mean():.6f}")


Exportando embeddings normalizados...

✅ Embeddings salvos com sucesso!
   user_embeddings.npy: (610, 64)
   movie_embeddings.npy: (9314, 64)
   user2idx.pkl: 610 usuários
   movie2idx.pkl: 9314 filmes
   embeddings_metadata.json: metadados

📊 Verificação de Normalização:
   User embeddings normas (deve ser ~1.0): min=1.000000, max=1.000000, mean=1.000000
   Movie embeddings normas (deve ser ~1.0): min=1.000000, max=1.000000, mean=1.000000


In [14]:
# ===========================
# 12. Teste de Compatibilidade com app.py
# ===========================
import json
# Testa se conseguimos carregar os arquivos
print("Testando compatibilidade com app.py...\n")

# Carrega embeddings
user_emb_test = np.load("user_embeddings.npy")
movie_emb_test = np.load("movie_embeddings.npy")

# Carrega mapeamentos
with open("user2idx.pkl", "rb") as f:
    u2i_test = pickle.load(f)

with open("movie2idx.pkl", "rb") as f:
    m2i_test = pickle.load(f)

# Carrega metadados
with open("embeddings_metadata.json", "r") as f:
    meta_test = json.load(f)

print(f"✅ user_embeddings.npy carregado: {user_emb_test.shape}")
print(f"✅ movie_embeddings.npy carregado: {movie_emb_test.shape}")
print(f"✅ user2idx.pkl carregado: {len(u2i_test)} usuários")
print(f"✅ movie2idx.pkl carregado: {len(m2i_test)} filmes")
print(f"✅ embeddings_metadata.json carregado: {meta_test}")

# Teste de busca por similaridade
print("\n" + "="*50)
print("TESTE: Busca por similaridade (Cosseno)")
print("="*50)

# Pega embedding de um usuário aleatório
test_user_idx = np.random.randint(0, len(user_emb_test))
test_user_emb = user_emb_test[test_user_idx]  # (emb_dim,)

# Calcula similaridade com todos os filmes
similarities = movie_emb_test @ test_user_emb  # (n_movies,)

# Top-5 filmes mais similares
top5_indices = np.argsort(similarities)[::-1][:5]

print(f"\nTop 5 filmes mais similares para usuário {test_user_idx}:")
for rank, movie_idx in enumerate(top5_indices, 1):
    sim = similarities[movie_idx]
    print(f"  {rank}. Movie {movie_idx}: {sim:.4f}")

print("\n✅ Teste de compatibilidade PASSOU!")
print("\n📝 Arquivos gerados:")
print("   - user_embeddings.npy")
print("   - movie_embeddings.npy")
print("   - user2idx.pkl")
print("   - movie2idx.pkl")
print("   - embeddings_metadata.json")
print("\n💡 Seu app.py pode usar esses arquivos sem modificações!")


Testando compatibilidade com app.py...

✅ user_embeddings.npy carregado: (610, 64)
✅ movie_embeddings.npy carregado: (9314, 64)
✅ user2idx.pkl carregado: 610 usuários
✅ movie2idx.pkl carregado: 9314 filmes
✅ embeddings_metadata.json carregado: {'n_users': 610, 'n_movies': 9314, 'emb_dim': 64, 'n_genres': 20, 'tag_dim': 384, 'normalization': 'L2 (cosine similarity)'}

TESTE: Busca por similaridade (Cosseno)

Top 5 filmes mais similares para usuário 101:
  1. Movie 2014: 0.8017
  2. Movie 8358: 0.7974
  3. Movie 9273: 0.7751
  4. Movie 4090: 0.7713
  5. Movie 7870: 0.7706

✅ Teste de compatibilidade PASSOU!

📝 Arquivos gerados:
   - user_embeddings.npy
   - movie_embeddings.npy
   - user2idx.pkl
   - movie2idx.pkl
   - embeddings_metadata.json

💡 Seu app.py pode usar esses arquivos sem modificações!
